#  Knowledge Graph Extraction Pipeline
**Objectif** : Extraire les entités et relations de documents texte, puis les stocker dans Neo4j.

---
### Étapes :
1.  Connexion au LLM (NVIDIA Mistral Large)
2. Chargement des documents
3. Extraction des entités et relations
4.  Fusion des résultats
5. Génération des requêtes Cypher
6.  Envoi vers Neo4j

##  1 — Imports et configuration

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Charger les variables d'environnement (.env)
load_dotenv()

print(f" NVIDIA_API_KEY chargée : {'✓' if os.getenv('NVIDIA_API_KEY') else '✗ MANQUANTE'}")
print(f"  NEO4J_URI : {os.getenv('NEO4J_URI', 'Non défini')}")


 NVIDIA_API_KEY chargée : ✓
  NEO4J_URI : bolt://localhost:7687


###  2 — Initialisation du LLM

In [2]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=32000,
)


## 3 — Test de connexion

In [3]:
response = llm.invoke("Reply with exactly: CONNECTION OK")
print("[Connection test]", response.content.strip())

[Connection test] CONNECTION OK


##  4 — Définition du schéma du graphe

In [4]:
SCHEMA = """
NODES (entity: fields):
  Company         : name, activity, certifications
  Supplier        : name
  Agreement       : name, description, type, startdate, enddate
  Document        : reference, title, version, date, owner
  Clients         : name, sector
  Clause          : name, description
  governance_body : name, acronyme, role
  Claim           : claim_type, value, unit, scope, source_doc, source_doc_id, quote
  Roles           : title
  Asset           : type, description, classification
  Algorithm       : name, use
  Protocol        : name, use, source
  Risk            : description, level
  Framework       : name, type, use
  Control         : name, requirement, source, Cycle
  Technology      : name, use, source

RELATIONSHIPS (source -[REL]-> target):
  Agreement       -[for]->              Supplier
  Company         -[Use]->              Agreement
  Supplier        -[HAS]->              Company
  Company         -[HAS]->              Document
  Company         -[HAS]->              Clients
  Company         -[HAS]->              governance_body
  Company         -[has_requirement]->  Clause
  Supplier        -[apply_to]->         Clause
  governance_body -[supervise]->        Roles
  governance_body -[approves]->         Claim
  Claim           -[Concerns]->         Asset
  Claim           -[Asserted_by]->      Roles
  Roles           -[operates]->         Technology
  Roles           -[EXECUTES]->         Control
  Algorithm       -[protects]->         Asset
  Protocol        -[used_by]->          Algorithm
  Protocol        -[mitigates]->        Risk
  Risk            -[targeting]->        Asset
  Framework       -[required_by]->      Control
  Control         -[IMPLEMENTED_BY]->   Technology
  Technology      -[Has_access_to]->    Asset
"""

SYSTEM_PROMPT = f"""
You are a knowledge-graph extraction assistant.
Given a text document, extract ALL entities and relationships that match
the schema below. Be exhaustive — extract every entity and relationship you can find.
Return ONLY a valid JSON object, no markdown, no explanation.

{SCHEMA}

IMPORTANT HINTS:
- Any vendor, contractor, or third-party provider → extract as Supplier
- Any contract, MSA, SLA, agreement → extract as Agreement
- Agreement links to Supplier with [for] relation
- Company links to Agreement with [Use] relation
- Supplier links to Clause with [apply_to] relation
- Any threat, attack, or vulnerability targeting an asset → extract Risk -[targeting]-> Asset

Output format:
{{
  "entities": [
    {{"label": "<NodeLabel>", "properties": {{"field": "value"}}}}
  ],
  "relationships": [
    {{
      "from_label": "<NodeLabel>",
      "from_key":   {{"field": "value"}},
      "rel_type":   "<REL_TYPE>",
      "to_label":   "<NodeLabel>",
      "to_key":     {{"field": "value"}}
    }}
  ]
}}

Rules:
- Use ONLY node labels and relationship types from the schema.
- For from_key / to_key use the single most identifying property.
- If a value is unknown, omit the field.
- String values only, no nested objects.
"""
print(" Schéma défini :")
print(f"   - 16 types de nœuds")
print(f"   - 21 types de relations")

 Schéma défini :
   - 16 types de nœuds
   - 21 types de relations


## 5 — Fonction d'extraction

In [5]:
import json
import time
from concurrent.futures import ThreadPoolExecutor

# ── Prompts séparés ───────────────────────────────────────────────

SYSTEM_PROMPT_ENTITIES = f"""You are a knowledge-graph extraction assistant.
Extract ONLY entities from the text using this schema:
{SCHEMA}
Return ONLY valid JSON, no markdown:
{{"entities":[{{"label":"<Label>","properties":{{"field":"value"}}}}]}}
Rules: one identifying property per entity, string values only."""

SYSTEM_PROMPT_RELATIONS = f"""You are a knowledge-graph extraction assistant.
Given a list of already extracted entities and a text document, extract ONLY the relationships between these entities using this schema:
{SCHEMA}
Return ONLY valid JSON, no markdown:
{{"relationships":[{{"from_label":"<Label>","from_key":{{"field":"value"}},"rel_type":"<REL>","to_label":"<Label>","to_key":{{"field":"value"}}}}]}}
Rules: use only relationship types from the schema, one identifying property for keys, string values only."""

# ── Découpage par === DOCUMENT === ────────────────────────────────

def split_by_document(text: str) -> list:
    """Découpe par marqueur === DOCUMENT X ==="""
    sections = []
    parts = text.split("=== DOCUMENT")
    for part in parts[1:]:
        sections.append("=== DOCUMENT" + part.strip())
    return sections if sections else [text]

# ── Extraction en 2 passes ────────────────────────────────────────

def extract_entities_only(document_text: str) -> list:
    """Passe 1 : extraire uniquement les entités."""
    messages = [
        ("system", SYSTEM_PROMPT_ENTITIES),
        ("human", f"Document:\n\n{document_text}"),
    ]
    raw = llm.invoke(messages).content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw).get("entities", [])


def extract_relations_only(document_text: str, entities: list) -> list:
    """Passe 2 : extraire les relations en donnant les entités déjà trouvées."""
    entities_str = json.dumps(entities, ensure_ascii=False)
    messages = [
        ("system", SYSTEM_PROMPT_RELATIONS),
        ("human", f"Already extracted entities:\n{entities_str}\n\nDocument:\n\n{document_text}"),
    ]
    raw = llm.invoke(messages).content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw).get("relationships", [])


def extract_entities_with_retry(text, retries=3):
    for attempt in range(retries):
        try:
            return extract_entities_only(text)
        except Exception as e:
            if "504" in str(e) and attempt < retries - 1:
                wait = (attempt + 1) * 15
                print(f"         timeout, retry {attempt+1}/{retries} dans {wait}s...")
                time.sleep(wait)
            else:
                return []

def extract_relations_with_retry(text, entities, retries=3):
    for attempt in range(retries):
        try:
            return extract_relations_only(text, entities)
        except Exception as e:
            if "504" in str(e) and attempt < retries - 1:
                wait = (attempt + 1) * 15
                print(f"         timeout, retry {attempt+1}/{retries} dans {wait}s...")
                time.sleep(wait)
            else:
                return []


def merge_results(results: list) -> dict:
    merged_entities, merged_relations = [], []
    seen_entities, seen_relations = set(), set()
    for result in results:
        for entity in result.get("entities", []):
            key = (entity["label"], str(entity.get("properties", {})))
            if key not in seen_entities:
                seen_entities.add(key)
                merged_entities.append(entity)
        for rel in result.get("relationships", []):
            key = (rel.get("from_label"), str(rel.get("from_key")),
                   rel.get("rel_type"), rel.get("to_label"), str(rel.get("to_key")))
            if key not in seen_relations:
                seen_relations.add(key)
                merged_relations.append(rel)
    return {"entities": merged_entities, "relationships": merged_relations}


def process_document(args):
    """Traite 1 document en 2 passes : entités puis relations."""
    i, section, total = args
    title = section.split("\n")[0].strip()
    print(f"      doc {i+1}/{total} : {title}")

    # Passe 1 — entités
    entities = extract_entities_with_retry(section)
    print(f"         -> {len(entities)} entites")

    # Passe 2 — relations
    relations = extract_relations_with_retry(section, entities)
    print(f"         -> {len(relations)} relations")

    return i, {"entities": entities, "relationships": relations}


def extract_full_document(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    print(f"   {len(text):,} caracteres")

    sections = split_by_document(text)
    print(f"   {len(sections)} documents — traitement parallele (3 a la fois)")

    results = [None] * len(sections)
    args = [(i, section, len(sections)) for i, section in enumerate(sections)]

    with ThreadPoolExecutor(max_workers=3) as executor:
        for i, result in executor.map(process_document, args):
            results[i] = result

    return merge_results(results)

print("Fonctions pretes — extraction en 2 passes par document")

Fonctions pretes — extraction en 2 passes par document


##  6 — Chargement des documents

In [6]:
DOCUMENTS = [
    "/home/maroua/Desktop/cassiope/SecuraOps High.txt",
    "/home/maroua/Desktop/cassiope/SafeLink Contradictory.txt",
]

documents_text = {}
for filepath in DOCUMENTS:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    documents_text[filepath] = text
    print(f" {filepath} : {len(text):,} caractères chargés")

 /home/maroua/Desktop/cassiope/SecuraOps High.txt : 39,933 caractères chargés
 /home/maroua/Desktop/cassiope/SafeLink Contradictory.txt : 21,896 caractères chargés


In [7]:
raw_results = {}

for filepath in DOCUMENTS:
    nom = filepath.split("/")[-1].replace(".txt", "")
    print(f"\n Extraction : {nom}")
    result = extract_full_document(filepath)  
    raw_results[filepath] = result

    with open(f"{nom}_graph.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"   Total : {len(result['entities'])} entités, {len(result['relationships'])} relations")
    print(f"   Sauvegardé : {nom}_graph.json")


 Extraction : SecuraOps High
   39,933 caracteres
   44 documents — traitement parallele (3 a la fois)
      doc 1/44 : === DOCUMENT1 : Politique de Sécurité de l’Information ===
      doc 2/44 : === DOCUMENT2 : Politique de Gestion des Accès ===
      doc 3/44 : === DOCUMENT3 : Politique de Gestion des Mots de Passe ===
         -> 1 entites
         -> 0 relations
      doc 4/44 : === DOCUMENT4 : Politique de Classification des Données ===
         -> 1 entites
         -> 0 relations
      doc 5/44 : === DOCUMENT5 : Politique de Sauvegarde et Restauration ===
         -> 1 entites
         -> 0 relations
      doc 6/44 : === DOCUMENT6 : Politique de Sécurité Réseau et Périmètre ===
         -> 1 entites
         -> 1 entites
         -> 0 relations
      doc 7/44 : === DOCUMENT7 : Politique de Gestion des Vulnérabilités ===
         -> 0 relations
      doc 8/44 : === DOCUMENT8 : Plan de Réponse aux Incidents de Sécurité ===
         -> 2 entites
         -> 1 entites
         -> 0

In [8]:
# Aperçu des résultats pour chaque document
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1]
    print(f"\n{nom}")
    print(f"   Entités   : {len(result['entities'])}")
    print(f"   Relations : {len(result['relationships'])}")
    
    # Résumé par type de nœud
    from collections import Counter
    node_counts = Counter(e['label'] for e in result['entities'])
    print("   Types d'entités :")
    for label, count in sorted(node_counts.items()):
        print(f"      {label:<20} : {count}")


SecuraOps High.txt
   Entités   : 373
   Relations : 445
   Types d'entités :
      Agreement            : 2
      Algorithm            : 8
      Asset                : 41
      Certification        : 2
      Claim                : 7
      Clause               : 21
      Clients              : 4
      Company              : 9
      Control              : 41
      Document             : 51
      Framework            : 16
      Protocol             : 23
      Risk                 : 8
      Roles                : 41
      Supplier             : 2
      Technology           : 82
      governance_body      : 15

SafeLink Contradictory.txt
   Entités   : 201
   Relations : 183
   Types d'entités :
      Agreement            : 1
      Algorithm            : 1
      Asset                : 22
      Certification        : 4
      Claim                : 20
      Clause               : 8
      Clients              : 6
      Company              : 9
      Control              : 15
      Document  

In [10]:
# Bloc 10 — Sauvegarde JSON pour chaque document
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    output_file = f"{nom}_graph1.json"
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    
    print(f" Sauvegardé : {output_file}")

 Sauvegardé : SecuraOps High_graph1.json
 Sauvegardé : SafeLink Contradictory_graph1.json


In [14]:
def props_str(props: dict) -> str:
    parts = []
    for k, v in props.items():
        safe_v = str(v).replace("'", "\\'")
        parts.append(f"{k}: '{safe_v}'")
    return "{" + ", ".join(parts) + "}"

def json_to_cypher(extracted: dict, company_label: str = None) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            if company_label:
                statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
            else:
                statements.append(f"MERGE (n:{label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        if not rel.get("from_key") or not rel.get("to_key") or not rel.get("rel_type"):
            continue
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        statements.append(
            f"MATCH (a:{fl} {fk}), (b:{tl} {tk}) MERGE (a)-[:{rt}]->(b);"
        )
    return statements

company_map = {
    "SecuraOps High": "SecuraOps",
    "SafeLink Contradictory": "SafeLink",
}

for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher(result, company_label=company)
    print(f"{nom} : {len(stmts)} statements")

SecuraOps High : 811 statements
SafeLink Contradictory : 380 statements


In [15]:
# Sauvegarde des requêtes Cypher pour chaque entreprise
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher(result, company)
    
    output_file = f"{nom}_cypher1.cypher"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(stmts))
    
    print(f"Sauvegardé : {output_file} ({len(stmts)} statements)")

Sauvegardé : SecuraOps High_cypher1.cypher (811 statements)
Sauvegardé : SafeLink Contradictory_cypher1.cypher (380 statements)


In [ ]:
c

Base videe


NameError: name 'raw_results' is not defined

In [4]:
DOCUMENTS2 = [
    "/home/maroua/Desktop/cassiope/NeoTrust Ambiguous.txt",
    "/home/maroua/Desktop/cassiope/CloudFirst Low.txt",
]

documents_text = {}
for filepath in DOCUMENTS2:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    documents_text[filepath] = text
    print(f" {filepath} : {len(text):,} caractères chargés")

 /home/maroua/Desktop/cassiope/NeoTrust Ambiguous.txt : 51,571 caractères chargés
 /home/maroua/Desktop/cassiope/CloudFirst Low.txt : 20,286 caractères chargés


In [14]:
raw_results = {}

for filepath in DOCUMENTS2:
    nom = filepath.split("/")[-1].replace(".txt", "")
    print(f"\n Extraction : {nom}")
    result = extract_full_document(filepath)  
    raw_results[filepath] = result

    with open(f"{nom}_graph.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"   Total : {len(result['entities'])} entités, {len(result['relationships'])} relations")
    print(f"   Sauvegardé : {nom}_graph.json")


 Extraction : NeoTrust Ambiguous
    51,571 caracteres
    21 sections — traitement parallele (3 a la fois)
      section 2/21 (2,578 chars) — 32 entites
      section 1/21 (2,578 chars) — 29 entites
      section 4/21 (2,578 chars) — 24 entites
      section 6/21 (2,578 chars) — 35 entites
      section 3/21 — timeout, retry 1/3 dans 15s...
      section 5/21 — timeout, retry 1/3 dans 15s...
      section 3/21 (2,578 chars) — 19 entites
      section 7/21 — timeout, retry 1/3 dans 15s...
      section 5/21 (2,578 chars) — 24 entites
      section 8/21 (2,578 chars) — 22 entites
      section 9/21 (2,578 chars) — 30 entites
      section 11/21 (2,578 chars) — 30 entites
      section 12/21 (2,578 chars) — 40 entites
      section 7/21 (2,578 chars) — 22 entites
      section 10/21 (2,578 chars) — 37 entites
      section 14/21 (2,578 chars) — 38 entites
      section 13/21 — timeout, retry 1/3 dans 15s...
      section 15/21 — timeout, retry 1/3 dans 15s...
      section 16/21 — timeo

In [15]:
# Aperçu des résultats pour chaque document
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1]
    print(f"\n{nom}")
    print(f"   Entités   : {len(result['entities'])}")
    print(f"   Relations : {len(result['relationships'])}")
    
    # Résumé par type de nœud
    from collections import Counter
    node_counts = Counter(e['label'] for e in result['entities'])
    print("   Types d'entités :")
    for label, count in sorted(node_counts.items()):
        print(f"      {label:<20} : {count}")


NeoTrust Ambiguous.txt
   Entités   : 573
   Relations : 496
   Types d'entités :
      Agreement            : 10
      Algorithm            : 5
      Asset                : 120
      Claim                : 8
      Clause               : 17
      Clients              : 7
      Company              : 9
      Control              : 90
      Cycle                : 2
      Document             : 53
      Framework            : 15
      Process              : 2
      Protocol             : 10
      Risk                 : 56
      Roles                : 47
      Supplier             : 12
      Technology           : 91
      governance_body      : 19

CloudFirst Low.txt
   Entités   : 260
   Relations : 276
   Types d'entités :
      Agreement            : 2
      Algorithm            : 1
      Asset                : 52
      Clause               : 5
      Clients              : 4
      Company              : 9
      Control              : 17
      Document             : 28
      Framework 

In [16]:
# Bloc 10 — Sauvegarde JSON pour chaque document
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    output_file = f"{nom}_graph.json"
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    
    print(f" Sauvegardé : {output_file}")

 Sauvegardé : NeoTrust Ambiguous_graph.json
 Sauvegardé : CloudFirst Low_graph.json


In [17]:
def props_str(props: dict) -> str:
    parts = []
    for k, v in props.items():
        safe_v = str(v).replace("'", "\\'")
        parts.append(f"{k}: '{safe_v}'")
    return "{" + ", ".join(parts) + "}"

def json_to_cypher(extracted: dict, company_label: str = None) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            if company_label:
                statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
            else:
                statements.append(f"MERGE (n:{label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        # Ignorer les relations incompletes
        if not rel.get("from_key") or not rel.get("to_key"):
            continue
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        statements.append(
            f"MATCH (a:{fl} {fk}), (b:{tl} {tk}) MERGE (a)-[:{rt}]->(b);"
        )
    return statements

company_map = {
    "NeoTrust Ambiguous": "NeoTrust",
    "CloudFirst Low": "CloudFirst",
}

for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher(result, company_label=company)
    print(f"{nom} : {len(stmts)} statements")

NeoTrust Ambiguous : 1069 statements
CloudFirst Low : 536 statements


In [18]:
# Sauvegarde des requêtes Cypher pour chaque entreprise
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher(result, company)
    
    output_file = f"{nom}_cypher.cypher"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(stmts))
    
    print(f"Sauvegardé : {output_file} ({len(stmts)} statements)")

Sauvegardé : NeoTrust Ambiguous_cypher.cypher (1069 statements)
Sauvegardé : CloudFirst Low_cypher.cypher (536 statements)


In [18]:
from neo4j import GraphDatabase

def props_str(props: dict) -> str:
    parts = []
    for k, v in props.items():
        safe_v = str(v).replace("'", "\\'")
        parts.append(f"{k}: '{safe_v}'")
    return "{" + ", ".join(parts) + "}"

def json_to_cypher_with_label(extracted: dict, company_label: str) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        if not rel.get("from_key") or not rel.get("to_key") or not rel.get("rel_type"):
            continue
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        statements.append(
            f"MATCH (a:{fl}:{company_label} {fk}), (b:{tl}:{company_label} {tk}) "
            f"MERGE (a)-[:{rt}]->(b);"
        )
    return statements

# V2 = nouvelle extraction par document + 2 passes
company_map = {
    "SecuraOps High": "SecuraOpsV2",
    "SafeLink Contradictory": "SafeLinkV2",
}

uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))
# Pas de DETACH DELETE — on garde l'ancienne extraction

for filepath, result in raw_results.items():
    nom     = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts   = json_to_cypher_with_label(result, company)
    print(f"\nEnvoi de {company}...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception:
                errors += 1
    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"   {errors} erreurs")

driver.close()
print("\nTermine ! -> http://localhost:7474")
print("\nComparer dans Neo4j :")
print("V1 : MATCH (n:SecuraOps)-[r]->(m:SecuraOps) RETURN n,r,m LIMIT 100")
print("V2 : MATCH (n:SecuraOpsV2)-[r]->(m:SecuraOpsV2) RETURN n,r,m LIMIT 100")
print("\nChiffres : MATCH (n) WHERE n:SecuraOps OR n:SecuraOpsV2")
print("RETURN CASE WHEN n:SecuraOps THEN 'V1' ELSE 'V2' END AS version, count(n)")


Envoi de SecuraOpsV2...
   811 statements OK

Envoi de SafeLinkV2...
   380 statements OK

Termine ! -> http://localhost:7474

Comparer dans Neo4j :
V1 : MATCH (n:SecuraOps)-[r]->(m:SecuraOps) RETURN n,r,m LIMIT 100
V2 : MATCH (n:SecuraOpsV2)-[r]->(m:SecuraOpsV2) RETURN n,r,m LIMIT 100

Chiffres : MATCH (n) WHERE n:SecuraOps OR n:SecuraOpsV2
RETURN CASE WHEN n:SecuraOps THEN 'V1' ELSE 'V2' END AS version, count(n)


In [4]:
import os, json
from neo4j import GraphDatabase
from dotenv import load_dotenv
load_dotenv()

# ── Charger les JSON depuis le disque ─────────────────────────────
raw_results = {}
files = {
    "/home/maroua/Desktop/cassiope/SecuraOps High.txt":         "/home/maroua/Desktop/cassiope/SecuraOps High_graph.json",
    "/home/maroua/Desktop/cassiope/SafeLink Contradictory.txt": "/home/maroua/Desktop/cassiope/SafeLink Contradictory_graph.json",
}
for filepath, json_file in files.items():
    with open(json_file, encoding="utf-8") as f:
        raw_results[filepath] = json.load(f)
    print(f"Charge : {json_file.split('/')[-1]} — {len(raw_results[filepath]['entities'])} entites")

# ── Fonctions ─────────────────────────────────────────────────────
def props_str(props: dict) -> str:
    parts = []
    for k, v in props.items():
        safe_v = str(v).replace("'", "\\'")
        parts.append(f"{k}: '{safe_v}'")
    return "{" + ", ".join(parts) + "}"

def json_to_cypher_with_label(extracted: dict, company_label: str) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        if not rel.get("from_key") or not rel.get("to_key"):
            continue
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        statements.append(
            f"MATCH (a:{fl}:{company_label} {fk}), (b:{tl}:{company_label} {tk}) "
            f"MERGE (a)-[:{rt}]->(b);"
        )
    return statements

company_map = {
    "SecuraOps High": "SecuraOps",
    "SafeLink Contradictory": "SafeLink",
}

# ── Envoi Neo4j ───────────────────────────────────────────────────
uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Base videe")

for filepath, result in raw_results.items():
    nom     = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts   = json_to_cypher_with_label(result, company)
    print(f"\nEnvoi de {company}...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception:
                errors += 1
    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"   {errors} erreurs")

driver.close()
print("\nTermine ! -> http://localhost:7474")

Charge : SecuraOps High_graph.json — 570 entites
Charge : SafeLink Contradictory_graph.json — 320 entites
Base videe

Envoi de SecuraOps...
   1090 statements OK

Envoi de SafeLink...
   602 statements OK

Termine ! -> http://localhost:7474


In [8]:
import os, json
from neo4j import GraphDatabase
from dotenv import load_dotenv
load_dotenv()

# Charger les JSON
raw_results = {}
files = {
    "/home/maroua/Desktop/cassiope/NeoTrust Ambiguous.txt": "/home/maroua/Desktop/cassiope/NeoTrust Ambiguous_graph.json",
    "/home/maroua/Desktop/cassiope/CloudFirst Low.txt":     "/home/maroua/Desktop/cassiope/CloudFirst Low_graph.json",
}
for filepath, json_file in files.items():
    with open(json_file, encoding="utf-8") as f:
        raw_results[filepath] = json.load(f)
    print(f"Charge : {json_file.split('/')[-1]} — {len(raw_results[filepath]['entities'])} entites")

def props_str(props):
    parts = []
    for k, v in props.items():
        safe_v = str(v).replace("'", "\\'")
        parts.append(f"{k}: '{safe_v}'")
    return "{" + ", ".join(parts) + "}"

def json_to_cypher_with_label(extracted, company_label):
    statements = []
    for entity in extracted.get("entities", []):
        props = entity.get("properties", {})
        if props:
            statements.append(f"MERGE (n:{entity['label']}:{company_label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        if not rel.get("from_key") or not rel.get("to_key"):
            continue
        statements.append(
            f"MATCH (a:{rel['from_label']}:{company_label} {props_str(rel['from_key'])}), "
            f"(b:{rel['to_label']}:{company_label} {props_str(rel['to_key'])}) "
            f"MERGE (a)-[:{rel['rel_type']}]->(b);"
        )
    return statements

company_map = {
    "NeoTrust Ambiguous": "NeoTrust",
    "CloudFirst Low":     "CloudFirst",
}

uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

for filepath, result in raw_results.items():
    nom     = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts   = json_to_cypher_with_label(result, company)
    print(f"\nEnvoi de {company}...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception:
                errors += 1
    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"   {errors} erreurs")

driver.close()
print("\nTermine ! -> http://localhost:7474")

Charge : NeoTrust Ambiguous_graph.json — 573 entites
Charge : CloudFirst Low_graph.json — 260 entites

Envoi de NeoTrust...
   1069 statements OK

Envoi de CloudFirst...
   536 statements OK

Termine ! -> http://localhost:7474
